In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path
import json 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.gridspec as gridspec
import geopandas as gpd

import torch
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.config_models import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets/"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

BARPLOT_ALPHA = 0.7

In [ ]:
def to_roman(n):
    vals = [(1000, 'm'), (900, 'cm'), (500, 'd'), (400, 'cd'), (100, 'c'),
            (90, 'xc'), (50, 'l'), (40, 'xl'), (10, 'x'), (9, 'ix'), (5, 'v'),
            (4, 'iv'), (1, 'i')]
    result = ''
    for v, s in vals:
        while n >= v:
            result += s
            n -= v
    return result

## Datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
print(num_measurements)

SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

### Subregions:

In [ ]:
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]

In [ ]:
# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# With subregions
TARGET_REGIONS_SUB = [
    'IT', 'AT', 'SJM', 'CA_3', 'CA_4', 'ALA_2', 'ALA_4', 'ALA_6'
    #'CENTRALASIA', 'ALA', #'CA_12',
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

### Pooled set:

In [ ]:
# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["df_train"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=True,
).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses
glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=False,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

# Failsafe: check that pooled training data only contains source region codes
actual_codes = set(df_all_glaciers.region.unique())
expected_codes = set(SOURCE_REGIONS)
unexpected = actual_codes - expected_codes
if unexpected:
    raise ValueError(
        f"Pooled training set contains unexpected SOURCE_CODEs: {unexpected}")
else:
    print(f"✓ Pooled set SOURCE_CODEs OK: {actual_codes}")

print(f"Pooled model assets: {len(assets_full['ds_train'])} sequences")

# ── Always compute scalers/blurs — needed for any recomputation ───────────────
res_source_only = {
    region: {
        "df_train": res_xreg_by_source[region]["df_train"],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)
blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

### Hold-out set:

In [ ]:
RECOMPUTE_SPLITS = False

FT_FRAC = 0.10
save_path = BASE_DIR / "glacier_splits_FD.json"

#gl_level_split = ['IT', 'AT']
gl_level_split = []
id_level_split = ['IT', 'AT', 'SJM', 'CA_3', 'CA_4', 'ALA_2', 'ALA_4', 'ALA_6']

# ── Load existing glacier splits from disk ────────────────────────────────────
splits = {}
if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        splits = json.load(f)
    print(f"Loaded splits from: {save_path}")

# ── Only compute glacier splits for regions that need them ────────────────────
regions_needing_gl_split = [r for r in gl_level_split]
regions_to_compute = (regions_needing_gl_split if RECOMPUTE_SPLITS else
                      [r for r in regions_needing_gl_split if r not in splits])

for region in regions_to_compute:
    print(f"\n{'='*50}\nSplitting region: {region}")
    split = split_pool_holdout_sinkhorn(
        df_region=monthly_cache[region]["data_monthly"],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.2,
        n_restarts=50,
        seed=cfg.seed,
    )
    splits[region] = split
    print(f"  Pool glaciers    : {split['pool_glaciers']}")
    print(f"  Holdout glaciers : {split['holdout_glaciers']}")
    print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

if regions_to_compute:
    with open(save_path, "w") as f:
        json.dump(splits, f, indent=2)
    print(f"\nSaved splits for: {regions_to_compute}")

In [ ]:
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_REGIONS = []

N_REPEATS = 5

# ── Build FT assets for all target subregions, repeated N_REPEATS times ──────
ft_assets_glacier = {}
ft_assets_id = {region: [] for region in TARGET_REGIONS_SUB}

for region in TARGET_REGIONS_SUB:
    print(f"\n{'='*50}\nBuilding FT assets for: {region}")

    df_monthly = monthly_cache[region]["data_monthly"]
    df_monthly_aug = monthly_cache[region]["data_monthly_aug"]
    head_pad = monthly_cache[region]["months_head_pad"]
    tail_pad = monthly_cache[region]["months_tail_pad"]
    force = FORCE_RECOMPUTE or (region in FORCE_RECOMPUTE_REGIONS)

    # 1. Glacier-level split (unchanged — deterministic by glacier, no repeats needed)
    if region in gl_level_split:
        if region in gl_level_split:
            pool_glaciers = splits[region]["pool_glaciers"]
            holdout_glaciers = splits[region]["holdout_glaciers"]
            exp_key = f"FD_to_{region}"

            ds_ft_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_FT",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    pool_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_TEST",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(
                    holdout_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    holdout_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_gl, ft_val_idx_gl = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_gl), val_ratio=0.2, seed=cfg.seed)
            ft_assets_glacier[region] = {
                "ds_ft": ds_ft_gl,
                "ds_test": ds_test_gl,
                "ft_train_idx": ft_train_idx_gl,
                "ft_val_idx": ft_val_idx_gl,
            }
            print(
                f"  [glacier-split] ds_ft  : {len(ds_ft_gl)} seqs | ds_test: {len(ds_test_gl)} seqs"
            )

    # 2. ID-level split, repeated N_REPEATS times with different seeds
    if region in id_level_split:
        winter_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "winter"]["ID"].unique().tolist()
        annual_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "annual"]["ID"].unique().tolist()

        for rep in range(N_REPEATS):
            winter_ids = list(winter_ids_full)
            annual_ids = list(annual_ids_full)

            rep_seed = cfg.seed + rep
            rng = np.random.default_rng(rep_seed)
            rng.shuffle(winter_ids)
            rng.shuffle(annual_ids)

            n_ft_w = max(1, int(len(winter_ids) *
                                FT_FRAC)) if winter_ids else 0
            n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
            ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
            test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

            # keep track of ft_ids per rep for the cross-rep check below
            if rep == 0:
                _ft_id_sets_for_check = []
            _ft_id_sets_for_check.append(set(ft_ids))

            exp_key_id = f"FD_to_{region}_IDsplit_rep{rep}"
            ds_ft_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_FT",
                df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_TEST",
                df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_id, ft_val_idx_id = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_id), val_ratio=0.2, seed=rep_seed)

            ft_assets_id[region].append({
                "rep": rep,
                "seed": rep_seed,
                "ds_ft": ds_ft_id,
                "ds_test": ds_test_id,
                "ft_train_idx": ft_train_idx_id,
                "ft_val_idx": ft_val_idx_id,
            })
            print(f"  [ID-split rep {rep}] ds_ft  : {len(ds_ft_id)} seqs | "
                  f"ds_test: {len(ds_test_id)} seqs")

            # ── sanity check after all reps for this region are built ────────────
        if region in id_level_split:
            n_reps = len(_ft_id_sets_for_check)
            identical_pairs = [
                (i, j) for i in range(n_reps) for j in range(i + 1, n_reps)
                if _ft_id_sets_for_check[i] == _ft_id_sets_for_check[j]
            ]
            if identical_pairs:
                print(
                    f"  WARNING [{region}]: identical FT id-sets for repeats {identical_pairs}"
                )
            else:
                overlaps = [
                    len(_ft_id_sets_for_check[i] & _ft_id_sets_for_check[j]) /
                    len(_ft_id_sets_for_check[i]) for i in range(n_reps)
                    for j in range(i + 1, n_reps)
                ]
                print(
                    f"  [check] {region}: {n_reps} repeats distinct (mean overlap {np.mean(overlaps):.1%})"
                )

# convenience dict: glacier-split where available, ID-split (list of reps) for the rest
ft_assets = {
    **ft_assets_glacier,
    **{
        r: ft_assets_id[r]
        for r in ft_assets_id if r not in ft_assets_glacier
    },
}

## Model assets:

## Transformer:

### Zero shot:

In [ ]:
TRAIN_FLAG = False
models_dir = BASE_DIR / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_pooled, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=PARAMS_TRANSFORMER,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=True,
)

print(f"Pooled model saved to: {model_path}")

# Build scaler once outside the loop
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

print(f"\n{'='*60}")
print("Zero-shot evaluation of pooled model on target regions")
print(f"{'='*60}")

# ── glacier-split: single dict per region, unchanged ──────────────────────────
print(f"\n--- glacier-split ---")
for region in gl_level_split:
    ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ft_assets_glacier[region]["ds_test"])
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
    metrics, _ = model_pooled.evaluate_with_preds(device, test_dl,
                                                  ds_test_copy)
    print(f"  {region:15s}  RMSE_a={metrics['RMSE_annual']:.3f}  "
          f"RMSE_w={metrics['RMSE_winter']:.3f}  "
          f"R2_a={metrics['R2_annual']:.3f}  "
          f"R2_w={metrics['R2_winter']:.3f}")

# ── ID-split: list of reps per region, aggregate across reps ─────────────────
print(f"\n--- ID-split (mean ± std over {N_REPEATS} repeats) ---")
zs_id_metrics = {
}  # store per-region rep metrics for later reuse (e.g. plotting)

for region in id_level_split:
    rep_metrics = []
    for rep_data in ft_assets_id[region]:
        ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            rep_data["ds_test"])
        test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
        metrics, _ = model_pooled.evaluate_with_preds(device, test_dl,
                                                      ds_test_copy)
        rep_metrics.append(metrics)

    zs_id_metrics[region] = rep_metrics

    rmse_a = [m["RMSE_annual"] for m in rep_metrics]
    rmse_w = [m["RMSE_winter"] for m in rep_metrics]
    r2_a = [m["R2_annual"] for m in rep_metrics]
    r2_w = [m["R2_winter"] for m in rep_metrics]

    print(f"  {region:15s}  "
          f"RMSE_a={np.mean(rmse_a):.3f}±{np.std(rmse_a):.3f}  "
          f"RMSE_w={np.mean(rmse_w):.3f}±{np.std(rmse_w):.3f}  "
          f"R2_a={np.mean(r2_a):.3f}±{np.std(r2_a):.3f}  "
          f"R2_w={np.mean(r2_w):.3f}±{np.std(r2_w):.3f}")

#### Predict on RGI:

In [ ]:
def predict_grid_in_chunks(
    cfg,
    region_folder,
    model,
    ds_pooled_scaler,
    months_head_pad,
    months_tail_pad,
    MONTHLY_COLS,
    STATIC_COLS,
    device,
    out_dir,
    ds_cache_dir=None,
    glaciers_per_chunk=50,
    batch_size=512,
    glacier_filter=None,
    skip_existing=True,
    force_recompute_ds=False,
):
    """
    Predict over RGI grid cells in chunks, writing one parquet file per
    chunk, and caching each chunk's pristine MBSequenceDataset to disk.
    """
    basepath = os.path.join(cfg.dataPath, "RGI_v6", region_folder)
    parquet_dir = os.path.join(basepath, "monthly_parquets")
    os.makedirs(out_dir, exist_ok=True)

    if ds_cache_dir is None:
        ds_cache_dir = os.path.join(out_dir, "ds_cache")
    os.makedirs(ds_cache_dir, exist_ok=True)

    parquet_paths = sorted(glob.glob(os.path.join(parquet_dir, "**", "*.parquet"), recursive=True))
    if not parquet_paths:
        raise FileNotFoundError(f"No parquet files found in {parquet_dir}")

    if glacier_filter is not None:
        parquet_paths = [
            p for p in parquet_paths
            if os.path.splitext(os.path.basename(p))[0] in glacier_filter
        ]

    print(f"{len(parquet_paths)} glacier files to predict, in chunks of {glaciers_per_chunk}")
    chunks = [parquet_paths[i:i + glaciers_per_chunk] for i in range(0, len(parquet_paths), glaciers_per_chunk)]

    n_rows_total = 0
    written_paths = []

    for chunk_idx, chunk_paths in enumerate(tqdm(chunks, desc="Predicting grid chunks")):
        chunk_out_path = os.path.join(out_dir, f"chunk_{chunk_idx:05d}.parquet")
        ds_cache_path = os.path.join(ds_cache_dir, f"chunk_{chunk_idx:05d}_ds.joblib")

        if skip_existing and os.path.exists(chunk_out_path):
            written_paths.append(chunk_out_path)
            continue  # resumable: re-running skips chunks already done

        # ---- pristine dataset: load from cache if present, else build + cache ----
        if (not force_recompute_ds) and os.path.exists(ds_cache_path):
            ds_chunk = joblib.load(ds_cache_path)
        else:
            df_chunk = pd.concat([pd.read_parquet(p) for p in chunk_paths], ignore_index=True)
            if len(df_chunk) == 0:
                continue

            df_chunk["POINT_BALANCE"] = 0.0

            mbm.utils.seed_all(cfg.seed)
            ds_chunk = build_combined_LSTM_dataset(
                df_loss=df_chunk,
                df_full=df_chunk,
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                months_head_pad=months_head_pad,
                months_tail_pad=months_tail_pad,
                normalize_target=True,
                expect_target=True,
                show_progress=False,
            )

            # sanity: must be pristine before caching (same guard as build_or_load_lstm_dataset_only)
            if (ds_chunk.month_mean is not None) or (ds_chunk.static_mean is not None) or (ds_chunk.y_mean is not None):
                raise ValueError(f"chunk {chunk_idx}: newly built dataset unexpectedly has scalers set.")

            joblib.dump(ds_chunk, ds_cache_path, compress=3)
            del df_chunk

        # ---- scale + predict ----
        ds_chunk_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(ds_chunk)
        chunk_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_chunk_copy, ds_pooled_scaler, batch_size=batch_size, seed=cfg.seed
        )

        df_preds_chunk = model.predict_with_keys(device, chunk_dl, ds_chunk_copy, denorm=True)
        df_preds_chunk.to_parquet(chunk_out_path, index=False)

        n_rows_total += len(df_preds_chunk)
        written_paths.append(chunk_out_path)

        del ds_chunk, ds_chunk_copy, chunk_dl, df_preds_chunk
        gc.collect()

    print(f"Done. Wrote {n_rows_total} predicted rows across {len(written_paths)} chunk files -> {out_dir}")
    return written_paths

In [ ]:
chunk_files = predict_grid_in_chunks(
    cfg=cfg,
    region_folder="RGI_11_CentralEurope",
    model=model_pooled,
    ds_pooled_scaler=ds_pooled_scaler,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    device=device,
    out_dir=os.path.join(cfg.dataPath, "RGI_v6", "RGI_11_CentralEurope", "grid_predictions_RGI11"),
    glaciers_per_chunk=50,
)

In [ ]:
# peek at results so far, without waiting for the whole run to finish
done_so_far = sorted(glob.glob(os.path.join(cfg.dataPath, "RGI_v6", "RGI_11_CentralEurope", "grid_predictions_RGI11", "chunk_*.parquet")))
print(f"{len(done_so_far)} chunks completed so far")
pd.read_parquet(done_so_far[0])

### Fine tuning:

#### Simple fine tuning:

In [ ]:
models_dir_ft = BASE_DIR / "models" / "finetuned"
os.makedirs(models_dir_ft, exist_ok=True)

ft_models_gl = {}  # glacier-split
ft_models_id = {}  # ID-split, now nested by rep

# ── glacier-split: unchanged structure ────────────────────────────────────────
for region in gl_level_split:
    ft_models_gl[region] = {}
    print(f"\n{'='*60}")
    print(f"Fine-tuning on: {region} (GLsplit)")

    for strategy in ["head_only", "full", "adapter"]:
        ft_models_gl[region][strategy] = finetune_transformer_on_target(
            cfg=cfg,
            model_pooled=model_pooled,
            ds_ft=ft_assets_glacier[region]["ds_ft"],
            ds_pooled_scaler=ds_pooled_scaler,
            device=device,
            best_params=PARAMS_TRANSFORMER,
            model_filename=str(
                models_dir_ft /
                f"transformer_ft_{strategy}_{region}_{MODEL_DATE}.pt"),
            strategy=strategy,
            epochs=PARAMS_FT['epochs'],
            lr_factor=PARAMS_FT['lr_factor'],
            force_retrain=False,
        )
        print(f"  [{region}] {strategy} done")

# ── ID-split: loop over reps, one model set per rep ──────────────────────────
for region in id_level_split:
    ft_models_id[region] = []  # list of per-rep strategy dicts
    print(f"\n{'='*60}")
    print(f"Fine-tuning on: {region} (IDsplit, {N_REPEATS} repeats)")

    for rep_data in ft_assets_id[region]:
        rep = rep_data["rep"]
        rep_models = {}

        for strategy in ["head_only", "full", "adapter"]:
            rep_models[strategy] = finetune_transformer_on_target(
                cfg=cfg,
                model_pooled=model_pooled,
                ds_ft=rep_data["ds_ft"],
                ds_pooled_scaler=ds_pooled_scaler,
                device=device,
                best_params=PARAMS_TRANSFORMER,
                model_filename=str(
                    models_dir_ft /
                    f"transformer_ft_{strategy}_{region}_IDsplit_rep{rep}_{MODEL_DATE}.pt"
                ),
                strategy=strategy,
                epochs=PARAMS_FT['epochs'],
                lr_factor=PARAMS_FT['lr_factor'],
                force_retrain=False,
            )
            print(f"  [{region}] rep {rep} - {strategy} done")

        ft_models_id[region].append({"rep": rep, "models": rep_models})

#### DAN fine tuning:

In [ ]:
source_codes_pretrain = build_source_codes_for_dataset(
    assets_full["ds_train"],
    pd.concat(
        [res_xreg_by_source[r]["df_train"] for r in SOURCE_REGIONS],
        ignore_index=True,
    ),
    source_col="SOURCE_CODE",
)

dan_models = {}  # glacier-split
dan_models_id = {}  # ID-split, now nested by rep

for region in id_level_split:
    print(f"\n{'='*50}  DAN → {region} (IDsplit, {N_REPEATS} repeats)")

    dan_models_id[region] = []  # list of per-rep DAN models

    for rep_data in ft_assets_id[region]:
        rep = rep_data["rep"]

        source_codes_ft = [region] * len(rep_data["ds_ft"])

        dan_model, _ = finetune_dan_transformer_on_target(
            cfg=cfg,
            model_foundation=model_pooled,
            assets_full=assets_full,
            ft_assets_region=rep_data,
            ds_pooled_scaler=ds_pooled_scaler,
            source_codes_pretrain=source_codes_pretrain,
            source_codes_ft=source_codes_ft,
            device=device,
            best_params=PARAMS_TRANSFORMER,
            model_filename=str(
                models_dir_ft /
                f"transformer_dan_{region}_IDsplit_rep{rep}_{MODEL_DATE}.pt"),
            dan_alpha=PARAMS_DAN['dan_alpha'],
            grl_lambda=PARAMS_DAN['grl_lambda'],
            mix_ratio_ft=PARAMS_DAN['mix_ratio_ft'],
            epochs=PARAMS_DAN['epochs'],
            lr_backbone=PARAMS_DAN['lr_backbone'],
            lr_domain=PARAMS_DAN['lr_domain'],
            force_retrain=False,
            verbose=False,
        )

        dan_models_id[region].append({"rep": rep, "model": dan_model})
        print(f"  [{region}] rep {rep} DAN done")

In [ ]:
# --- unified evaluation --- ID-split only, aggregated over repeats
print(f"\n{'='*75}")
print(f"{'Strategy':<25} {'RMSE_a':>8} {'RMSE_w':>8} {'R2_a':>8} {'R2_w':>8}")
print(f"{'='*75}")

df_ft_metrics_id = {}  # {region: {strategy_label: {metric: mean_value}}}
df_ft_metrics_id_raw = {
}  # {region: {strategy_label: [metrics_per_rep]}} for std/error bars

for region in id_level_split:
    print(f"\n--- {region} (mean over {N_REPEATS} repeats) ---")

    df_ft_metrics_id_raw[region] = {
        label: []
        for label in ["no_ft", "head_only", "adapter", "full", "dan"]
    }

    for rep_idx, rep_data in enumerate(ft_assets_id[region]):
        rep = rep_data["rep"]

        def _eval(model, ds_test=rep_data["ds_test"]):
            ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                ds_test)
            test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
            return model.evaluate_with_preds(device, test_dl, ds_test_copy)

        # models for this rep
        rep_ft_models = ft_models_id[region][rep_idx]["models"]
        rep_dan_model = dan_models_id[region][rep_idx]["model"]

        all_models = {
            "no_ft": model_pooled,
            "head_only": rep_ft_models["head_only"],
            "adapter": rep_ft_models["adapter"],
            "full": rep_ft_models["full"],
            "dan": rep_dan_model,
        }

        for label, model in all_models.items():
            m, _ = _eval(model)
            df_ft_metrics_id_raw[region][label].append(m)

    # aggregate across reps
    df_ft_metrics_id[region] = {}
    for label, rep_metrics_list in df_ft_metrics_id_raw[region].items():
        agg = {}
        for metric_key in rep_metrics_list[0].keys():
            vals = [m[metric_key] for m in rep_metrics_list]
            agg[metric_key] = float(np.mean(vals))
            agg[f"{metric_key}_std"] = float(np.std(vals))
        df_ft_metrics_id[region][label] = agg

        print(f"  {label:<25}  "
              f"RMSE_a={agg['RMSE_annual']:.3f}±{agg['RMSE_annual_std']:.3f}  "
              f"RMSE_w={agg['RMSE_winter']:.3f}±{agg['RMSE_winter_std']:.3f}  "
              f"R2_a={agg['R2_annual']:.3f}±{agg['R2_annual_std']:.3f}  "
              f"R2_w={agg['R2_winter']:.3f}±{agg['R2_winter_std']:.3f}")